
---
date: "2026-10-25"
date-modified: last-modified
format:
  html:
    toc: true
---


# Set Cover and MAX-SAT: Rounding Fractional Solutions

**Prerequisites:** [Vertex Cover and Independent Set](vertex-cover-and-independent-set.ipynb) · [Integer Programs and LP Relaxation](integer-programming.ipynb) · [Expectation](expectation.ipynb) · [Markov's Inequality](markovs-inequality.ipynb#thm-markovs-inequality)

## 1\. Set Cover

::: {#def-set-cover}
## Weighted Set Cover

Let $X = \{1, \dots, n\}$ and let $S_1, \dots, S_m \subseteq X$ with $S_1 \cup \dots \cup S_m = X$, with weights $w_j > 0$. A **cover** is a set $J \subseteq \{1, \dots, m\}$ with $\bigcup_{j \in J} S_j = X$; its weight is $w(J) = \sum_{j \in J} w_j$. The problem asks for a cover of minimum weight $\mathrm{OPT}$. With all $w_j = 1$ it asks for the fewest sets that cover $X$.
:::

::: {#prp-set-cover-ip}
## Set Cover as an Integer Program

$\mathrm{OPT}$ is the optimal value of
$$
\text{minimize } \sum_{j=1}^m w_jy_j \quad \text{subject to } \sum_{j :\, i \in S_j} y_j \ge 1\ \ (i = 1, \dots, n), \quad y \in \{0,1\}^m .
$$
Its relaxation ($0 \le y_j \le 1$) has an optimal solution $y^*$; let $\mathrm{LP}^* = \sum_j w_jy^*_j$. Then $0 < \mathrm{LP}^* \le \mathrm{OPT}$.
:::

::: {.callout-tip collapse="true"}
## Proof of @prp-set-cover-ip
For $y \in \{0,1\}^m$ let $J = \{j : y_j = 1\}$. The $i$-th constraint says that some $j \in J$ has $i \in S_j$, so the feasible $y$ are the indicators of covers, with $\sum w_jy_j = w(J)$. The relaxation has a nonempty ($y \equiv 1$), closed, bounded feasible set, so it has an optimum, and $\mathrm{LP}^* \le \mathrm{OPT}$ by [the relaxation bound](integer-programming.ipynb#prp-relaxation-bound). Finally $\mathrm{LP}^* > 0$: the constraint for element $1$ forces some $y^*_j > 0$, and $w_j > 0$. $\blacksquare$
:::

::: {#exm-vertex-cover-is-set-cover}
## Vertex Cover Is a Special Case

For a graph $G = (V, E)$, take $X = E$ and, for each vertex $v$, the set $S_v$ of edges containing $v$. A family $\{S_v : v \in C\}$ covers $E$ iff $C$ is a vertex cover. Each element (edge) lies in exactly $2$ of the sets. Since vertex cover is NP-hard, so is set cover.
:::

## 2\. Threshold Rounding

Let $\Delta = \max_i |\{j : i \in S_j\}|$ be the largest number of sets containing a single element.

::: {#thm-set-cover-threshold}
## A $\Delta$-Approximation

Let $J = \{j : y^*_j \ge \frac1\Delta\}$. Then $J$ is a cover and $w(J) \le \Delta\,\mathrm{LP}^* \le \Delta\,\mathrm{OPT}$.
:::

::: {.callout-tip collapse="true"}
## Proof of @thm-set-cover-threshold
Fix $i$. The constraint $\sum_{j : i \in S_j} y^*_j \ge 1$ has at most $\Delta$ terms, so the largest of them is $\ge \frac1\Delta$; that $j$ lies in $J$ and covers $i$. For $j \in J$, $1 \le \Delta y^*_j$, hence
$$
w(J) = \sum_{j \in J} w_j \le \sum_{j \in J} \Delta w_jy^*_j \le \Delta\sum_{j=1}^m w_jy^*_j = \Delta\,\mathrm{LP}^* . \qquad \blacksquare
$$
:::

For vertex cover $\Delta = 2$ and this is [rounding at $\frac12$](vertex-cover-and-independent-set.ipynb#thm-vertex-cover-rounding). In general $\Delta$ can be as large as $m$ (an element lying in every set), and the guarantee becomes weak.

## 3\. Randomized Rounding

The numbers $y^*_j \in [0, 1]$ can be used as probabilities: the larger $y^*_j$, the more likely $S_j$ is chosen.

::: {#lem-one-minus-t}
## An Elementary Inequality

$1 - t \le e^{-t}$ for all $t \in \mathbb{R}$.
:::

::: {.callout-tip collapse="true"}
## Proof of @lem-one-minus-t
Let $g(t) = e^{-t} - 1 + t$. Then $g(0) = 0$ and $g'(t) = 1 - e^{-t}$, which is $< 0$ for $t < 0$ and $> 0$ for $t > 0$. So $g$ decreases on $(-\infty, 0]$ and increases on $[0, \infty)$, and $g(t) \ge g(0) = 0$. $\blacksquare$
:::

::: {#lem-one-round}
## One Round of Randomized Rounding

Choose each index $j$ independently with probability $y^*_j$, and let $J_1$ be the set of chosen indices. Then

1. $\mathbb{E}[w(J_1)] = \mathrm{LP}^*$;
2. for every element $i$, $\Pr(i \text{ is not covered by } J_1) \le \frac1e$.

In particular $J_1$ covers on average at least a fraction $1 - \frac1e \approx 0.63$ of the elements.
:::

::: {.callout-tip collapse="true"}
## Proof of @lem-one-round

1. $w(J_1) = \sum_j w_j\mathbf 1[j \in J_1]$ and $\mathbb{E}\,\mathbf 1[j \in J_1] = y^*_j$; use linearity of [expectation](expectation.ipynb).
2. $i$ is not covered iff no $j$ with $i \in S_j$ is chosen. These choices are independent, so by @lem-one-minus-t
$$
\Pr(i \text{ not covered}) = \prod_{j :\, i \in S_j}(1 - y^*_j) \le \prod_{j :\, i \in S_j} e^{-y^*_j} = \exp\Big(-\sum_{j :\, i \in S_j} y^*_j\Big) \le e^{-1},
$$
using the constraint $\sum_{j : i \in S_j} y^*_j \ge 1$. The last statement follows by summing the probabilities $\Pr(i \text{ covered}) \ge 1 - \frac1e$ over $i$ (linearity again). $\blacksquare$
:::

::: {#thm-randomized-set-cover}
## Repeated Randomized Rounding

Let $n \ge 2$ and $k = \lceil 2\ln n\rceil$. Perform $k$ independent rounds as in @lem-one-round and let $J = J_1 \cup \dots \cup J_k$. Then

1. $\Pr(J \text{ is not a cover}) \le \frac1n \le \frac12$;
2. $\mathbb{E}[w(J)] \le k\,\mathrm{LP}^*$;
3. with probability at least $\frac14$, $J$ is a cover and $w(J) < 4k\,\mathrm{LP}^* \le 4k\,\mathrm{OPT}$.

Repeating the whole procedure $r$ times independently, and keeping a result that is a cover with $w(J) < 4k\,\mathrm{LP}^*$ (both conditions can be checked), fails with probability at most $(3/4)^r$. This gives a randomized $O(\log n)$-approximation, with no assumption on $\Delta$.
:::

::: {.callout-tip collapse="true"}
## Proof of @thm-randomized-set-cover

1. Element $i$ is uncovered by $J$ iff it is uncovered in every round. The rounds are independent, so by @lem-one-round this has probability $\le e^{-k} \le e^{-2\ln n} = n^{-2}$. The probability that some element is uncovered is at most the sum of these probabilities over the $n$ elements: $\le n \cdot n^{-2} = \frac1n$.
2. A set chosen in several rounds is counted once in $w(J)$, so $w(J) \le \sum_{r=1}^k w(J_r)$, and $\mathbb{E}[w(J)] \le k\,\mathrm{LP}^*$ by @lem-one-round.
3. $\mathrm{LP}^* > 0$, so [Markov's inequality](markovs-inequality.ipynb#thm-markovs-inequality) gives $\Pr\big(w(J) \ge 4k\,\mathrm{LP}^*\big) \le \frac{k\,\mathrm{LP}^*}{4k\,\mathrm{LP}^*} = \frac14$. The probability that $J$ is not a cover *or* $w(J) \ge 4k\,\mathrm{LP}^*$ is at most $\frac12 + \frac14$, so both good events happen with probability $\ge \frac14$; and $\mathrm{LP}^* \le \mathrm{OPT}$.

For the repetition, the attempts are independent and each fails with probability $\le \frac34$, so all $r$ fail with probability $\le (3/4)^r$. $\blacksquare$
:::

In [1]:

import numpy as np
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA, YELLOW, VIOLET, RED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6", "#e34948"
INK, MUTED, GRID, FAINT = "#0b0b0b", "#52514e", "#d9d8d4", "#a3a29a"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "legend.frameon": False, "font.size": 10})
from itertools import combinations
from scipy.optimize import linprog

rng = np.random.default_rng(7)
n, m = 18, 14
sets = [set(rng.choice(n, size=rng.integers(3, 7), replace=False)) for _ in range(m)]
for i in range(n):                                   # make sure every element is covered
    if not any(i in S for S in sets): sets[rng.integers(m)].add(i)
w = np.ones(m)
A = np.array([[1 if i in S else 0 for S in sets] for i in range(n)])
lp = linprog(w, A_ub=-A, b_ub=-np.ones(n), bounds=[(0, 1)] * m)
y, LP = lp.x, lp.fun
Delta = A.sum(axis=1).max()
is_cover = lambda J: set().union(*(sets[j] for j in J)) == set(range(n)) if J else False
OPT = next(k for k in range(1, m + 1) if any(is_cover(J) for J in combinations(range(m), k)))
J_thr = [j for j in range(m) if y[j] >= 1 / Delta - 1e-12]
print(f"LP* = {LP:.3f}, OPT = {OPT}, Delta = {Delta}")
print(f"threshold rounding: cover = {is_cover(J_thr)}, size {len(J_thr)} <= Delta*LP* = {Delta*LP:.2f}")

k = int(np.ceil(2 * np.log(n)))
trials, ok, sizes, frac1 = 2000, 0, [], []
for _ in range(trials):
    rounds = [set(np.flatnonzero(rng.random(m) < y)) for _ in range(k)]
    covered1 = set().union(*(sets[j] for j in rounds[0])) if rounds[0] else set()
    frac1.append(len(covered1) / n)
    J = set().union(*rounds)
    ok += is_cover(J); sizes.append(len(J))
print(f"one round covers on average {np.mean(frac1):.3f} of the elements (bound 1 - 1/e = {1 - np.exp(-1):.3f})")
print(f"k = {k} rounds: cover in {ok / trials:.3f} of the trials (bound 1 - 1/n = {1 - 1/n:.3f}); "
      f"mean size {np.mean(sizes):.2f} <= k*LP* = {k*LP:.2f}")

LP* = 3.750, OPT = 5, Delta = 6
threshold rounding: cover = True, size 8 <= Delta*LP* = 22.50
one round covers on average 0.788 of the elements (bound 1 - 1/e = 0.632)
k = 6 rounds: cover in 0.998 of the trials (bound 1 - 1/n = 0.944); mean size 8.17 <= k*LP* = 22.50



## 4\. MAX-SAT by a Random Assignment

Let $C_1 \wedge \dots \wedge C_m$ be a CNF formula and let $\mathrm{OPT}$ be the largest number of clauses that one assignment can satisfy (see [MAX-SAT](integer-programming.ipynb#prp-max-sat-ip)). We may assume that no clause contains the same variable twice: a repeated literal can be deleted, and a clause containing both $x_i$ and $\neg x_i$ is true under every assignment (remove it and add $1$ to the count). Let $k_j \ge 1$ be the number of literals of $C_j$.

::: {#thm-max-sat-random}
## A Random Assignment Satisfies Half of the Clauses on Average

Let $x_1, \dots, x_n$ be independent, each equal to $0$ or $1$ with probability $\frac12$. The expected number of satisfied clauses is
$$
\sum_{j=1}^m \big(1 - 2^{-k_j}\big) \ \ge\ \frac m2 \ \ge\ \frac{\mathrm{OPT}}2 .
$$
Hence some assignment satisfies at least $\frac m2$ clauses. If every clause has at least $k$ literals, some assignment satisfies at least $(1 - 2^{-k})\,m \ge (1 - 2^{-k})\,\mathrm{OPT}$ clauses; for $k = 4$ this is $\frac{15}{16}$ of the optimum.
:::

::: {.callout-tip collapse="true"}
## Proof of @thm-max-sat-random
$C_j$ is false iff each of its $k_j$ literals is false. These literals involve $k_j$ distinct variables, and each literal is false with probability $\frac12$; by independence $\Pr(C_j \text{ false}) = 2^{-k_j}$. Let $t_j = \mathbf 1[C_j \text{ true}]$; the number of satisfied clauses is $T = \sum_j t_j$ and by linearity $\mathbb{E}T = \sum_j (1 - 2^{-k_j})$. Since $k_j \ge 1$, each term is $\ge \frac12$; and $\mathrm{OPT} \le m$.

If every assignment satisfied fewer than $\mathbb{E}T$ clauses, then $T < \mathbb{E}T$ always, and so $\mathbb{E}T < \mathbb{E}T$, a contradiction. So some assignment satisfies at least $\mathbb{E}T$ clauses (compare the [probabilistic method](probabilistic-for-existence-proof.ipynb#thm-positive-probability-existence)). With $k_j \ge k$ the same argument gives the bound $(1 - 2^{-k})m$. $\blacksquare$
:::

::: {.callout-warning}
## Where Independence Is Used
Linearity of expectation, $\mathbb{E}\sum t_j = \sum \mathbb{E}t_j$, needs no independence: the $t_j$ are dependent when clauses share variables. Independence of the variables is used to compute each single probability $\Pr(C_j \text{ false}) = 2^{-k_j}$.
:::

The theorem proves that a good assignment exists but does not say how to find one. The randomness can be removed.

::: {#prp-max-sat-derandomized}
## Method of Conditional Expectations

For values $a_1, \dots, a_r \in \{0, 1\}$ of $x_1, \dots, x_r$, let $E(a_1, \dots, a_r)$ be the expected number of satisfied clauses when $x_1 = a_1, \dots, x_r = a_r$ and $x_{r+1}, \dots, x_n$ are random as above. Then

1. $E(a_1, \dots, a_r) = \frac12 E(a_1, \dots, a_r, 0) + \frac12 E(a_1, \dots, a_r, 1)$;
2. $E(a_1, \dots, a_r) = \sum_j e_j$, where $e_j = 1$ if $C_j$ already contains a true literal, and otherwise $e_j = 1 - 2^{-\ell_j}$ with $\ell_j$ the number of literals of $C_j$ on the unfixed variables.

Choosing $a_{r+1} \in \{0, 1\}$ with the larger value of $E(a_1, \dots, a_{r+1})$ for $r = 0, 1, \dots, n-1$ produces, in polynomial time, an assignment satisfying at least $E() = \sum_j(1 - 2^{-k_j}) \ge \frac m2$ clauses.
:::

::: {.callout-tip collapse="true"}
## Proof of @prp-max-sat-derandomized

1. Conditioning on the value of $x_{r+1}$, which is $0$ or $1$ with probability $\frac12$ each and independent of the other random variables.
2. As in @thm-max-sat-random: a clause with a true fixed literal is true; otherwise it is true unless all its $\ell_j$ unfixed literals are false, which has probability $2^{-\ell_j}$ (this is $1 - 2^{0} = 0$ when $\ell_j = 0$, i.e. when the clause is already false). Linearity gives the sum.

By 1, $\max\big(E(\dots, 0), E(\dots, 1)\big) \ge E(\dots)$, so the chosen values satisfy $E() \le E(a_1) \le \dots \le E(a_1, \dots, a_n)$. The last number is the number of clauses satisfied by the assignment $a$. By item 2, each value $E$ is computed in time proportional to the size of the formula. $\blacksquare$
:::

In [2]:

from itertools import product

rng = np.random.default_rng(4)
nv, mc = 12, 60
clauses = []
for _ in range(mc):
    k = rng.integers(1, 4)
    vars_ = rng.choice(nv, size=k, replace=False)
    clauses.append([(int(v), bool(rng.random() < 0.5)) for v in vars_])   # (variable, negated?)

def satisfied(a):
    return sum(any((a[v] == 0) if neg else (a[v] == 1) for v, neg in C) for C in clauses)

def cond_exp(a):            # a: dict of fixed variables
    tot = 0.0
    for C in clauses:
        if any(v in a and ((a[v] == 0) if neg else (a[v] == 1)) for v, neg in C):
            tot += 1
        else:
            free = sum(v not in a for v, _ in C)
            tot += 1 - 2.0 ** (-free)
    return tot

OPT = max(satisfied(a) for a in product([0, 1], repeat=nv))
expected = cond_exp({})
a = {}
for v in range(nv):
    a[v] = max([0, 1], key=lambda b: cond_exp({**a, v: b}))
greedy = satisfied([a[v] for v in range(nv)])
sim = np.mean([satisfied(rng.integers(0, 2, nv)) for _ in range(3000)])
print(f"m = {mc}, OPT = {OPT}, expected (formula) = {expected:.2f}, simulated mean = {sim:.2f}")
print(f"derandomized assignment satisfies {greedy} clauses (>= {expected:.2f} >= m/2 = {mc/2})")

m = 60, OPT = 51, expected (formula) = 42.88, simulated mean = 42.81
derandomized assignment satisfies 50 clauses (>= 42.88 >= m/2 = 30.0)
